# DCU 2026 ML Challenge — GenAI Image Binary Classification

**Goal:** Maximise binary classification **F1 score to 0.88–0.92** using:
- Transfer learning (ConvNeXt V2 Tiny + Swin Transformer Tiny)
- 512-d representation fusion from penultimate embeddings
- Meta-learner benchmarking (XGBoost · CatBoost · TabPFN)
- 5-view Test-Time Augmentation
- Per-learner threshold optimisation

> **Submission file:** `dcu_2026_final_submission.csv` — exact match to `test.csv` row order.

## Cell 0 — Setup + Challenge File Loading

Mount Google Drive, unzip the challenge image archive, load `train.csv` and `test.csv`,
inspect class distribution, build a robust image-path resolver, clean problematic
filenames (spaces, brackets, commas), configure seeds for full reproducibility,
detect CUDA, enable mixed-precision (AMP), and register a `free_memory()` helper
that is called aggressively throughout to stay within Colab free-tier RAM/VRAM limits.

In [ ]:
# ── 0.1  Install extra packages ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "xgboost", "pytorch-grad-cam", "catboost"], check=False)

# ── 0.2  Mount Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_CHALLENGE = "/content/drive/MyDrive/DCU 2026 ML Challenge"

# ── 0.3  Unzip challenge image archive ───────────────────────────────────────
import os, zipfile, shutil, re, gc, random, warnings, copy
from pathlib import Path
from collections import Counter

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_path = Path(DRIVE_CHALLENGE) / "genai_image_challenge.zip"
if zip_path.exists():
    print(f"Unzipping {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Done.")
else:
    raise FileNotFoundError(
        f"Challenge zip not found at {zip_path}. "
        "Check your Google Drive path."
    )

# ── 0.4  Standard imports ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T

import timm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve,
    average_precision_score,
)
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.contingency_tables import mcnemar

import xgboost as xgb
warnings.filterwarnings("ignore")

# ── 0.5  Reproducibility ─────────────────────────────────────────────────────
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── 0.6  Device & mixed-precision ────────────────────────────────────────────
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
scaler  = GradScaler(enabled=USE_AMP)
print(f"Device : {DEVICE}  |  AMP : {USE_AMP}")

# ── 0.7  Memory helper ────────────────────────────────────────────────────────
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ── 0.8  Load train.csv and test.csv ─────────────────────────────────────────
TRAIN_CSV = Path(DRIVE_CHALLENGE) / "train.csv"
TEST_CSV  = Path(DRIVE_CHALLENGE) / "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train.csv shape:", train_df.shape)
print("train.csv columns:", train_df.columns.tolist())
print(train_df.head(3))
print()
print("test.csv shape:", test_df.shape)
print("test.csv columns:", test_df.columns.tolist())
print(test_df.head(3))

# ── 0.9  Identify image-id and label columns ─────────────────────────────────
def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns {df.columns.tolist()}")

IMG_COL      = _find_col(train_df, ["image_id", "filename", "id", "image"])
LABEL_COL    = _find_col(train_df, ["label", "target", "class", "y"])
TEST_IMG_COL = _find_col(test_df,  ["image_id", "filename", "id", "image"])

print(f"\nImage column  : {IMG_COL!r}")
print(f"Label column  : {LABEL_COL!r}")
print(f"Test id column: {TEST_IMG_COL!r}")

# ── 0.10  Image path resolver ─────────────────────────────────────────────────
def _find_image_root(base):
    # Walk up to several levels looking for a directory holding image files
    for dirpath, dirnames, filenames in os.walk(base):
        imgs = [f for f in filenames
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
        if imgs:
            return Path(dirpath)
    return base

IMAGE_ROOT = _find_image_root(DATA_ROOT)
print(f"\nImage root detected: {IMAGE_ROOT}")

EXT_PRIORITIES = [".jpg", ".jpeg", ".png", ".webp"]

def resolve_image_path(img_id, root=None):
    if root is None:
        root = IMAGE_ROOT
    root = Path(root)
    stem = Path(str(img_id)).stem
    clean_stem = re.sub(r"[()\ ,]+", "_", stem)
    clean_stem = re.sub(r"_+", "_", clean_stem).strip("_")
    for ext in EXT_PRIORITIES:
        for s in [stem, clean_stem]:
            p = root / (s + ext)
            if p.exists():
                return p
    matches = list(root.glob(f"{clean_stem}*"))
    if matches:
        return matches[0]
    return root / str(img_id)

# ── 0.11  Clean problematic filenames ────────────────────────────────────────
def clean_filenames(root):
    renamed = 0
    for fpath in sorted(Path(root).rglob("*")):
        if not fpath.is_file():
            continue
        stem, suffix = fpath.stem, fpath.suffix
        new_stem = re.sub(r"[()\ ,]+", "_", stem)
        new_stem = re.sub(r"_+", "_", new_stem).strip("_")
        new_name = new_stem + suffix
        if new_name != fpath.name:
            dest = fpath.parent / new_name
            counter = 1
            while dest.exists():
                dest = fpath.parent / f"{new_stem}_{counter}{suffix}"
                counter += 1
            fpath.rename(dest)
            renamed += 1
    return renamed

n_renamed = clean_filenames(IMAGE_ROOT)
print(f"Filenames cleaned/renamed: {n_renamed}")

# ── 0.12  Class distribution & imbalance ratio ───────────────────────────────
label_counts    = train_df[LABEL_COL].value_counts().sort_index()
majority, minority = label_counts.max(), label_counts.min()
imbalance_ratio = majority / minority
print(f"\nClass distribution:\n{label_counts.to_string()}")
print(f"Imbalance ratio (majority/minority): {imbalance_ratio:.2f}")

plt.figure(figsize=(5, 3))
label_counts.plot(kind="bar", color=["steelblue", "coral"], edgecolor="black")
plt.title("Training Class Distribution")
plt.xlabel("Class"); plt.ylabel("Count"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# ── 0.13  Path-resolution integrity check ────────────────────────────────────
n_missing = 0
for img_id in train_df[IMG_COL].sample(min(50, len(train_df)), random_state=SEED):
    if not resolve_image_path(str(img_id)).exists():
        n_missing += 1

print(f"Sampled path resolution — missing: {n_missing}/50")
print(f"Total train: {len(train_df)}  |  Total test: {len(test_df)}")
print(f"Train label unique: {sorted(train_df[LABEL_COL].unique().tolist())}")
print("\n✓ Cell 0 complete.")

## Cell 1 — Dataset + Stratified Split

We build a custom `ChallengeDataset` class that reads image paths **directly from the
CSV files** rather than relying on `ImageFolder` directory conventions.  This makes the
pipeline robust to arbitrary directory layouts and gives precise control over which
images belong to which split.

**Training augmentations** (aggressive to prevent over-fitting):
horizontal flip · vertical flip · ±20° rotation · colour jitter · random grayscale ·
affine transform · centre-crop · ImageNet normalisation

The **80/20 stratified split** is applied to `train.csv` only.  
The challenge `test.csv` remains **completely untouched** until Cell 9.

In [ ]:
# ── 1.1  Hyper-parameters ─────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── 1.2  Transforms ───────────────────────────────────────────────────────────
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
    T.CenterCrop(int(IMG_SIZE * 0.95)),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── 1.3  CSV-driven dataset ───────────────────────────────────────────────────
from PIL import Image as PILImage

class ChallengeDataset(Dataset):
    # Reads image paths and labels from a pandas DataFrame.
    # label_col=None for the challenge test set (no ground-truth labels).
    def __init__(self, df, img_col, label_col, transform, img_root, mode="train"):
        self.df        = df.reset_index(drop=True)
        self.img_col   = img_col
        self.label_col = label_col
        self.transform = transform
        self.img_root  = Path(img_root)
        self.mode      = mode
        self.has_labels = (label_col is not None) and (label_col in df.columns)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img_id = str(row[self.img_col])
        path   = resolve_image_path(img_id, self.img_root)
        try:
            img = PILImage.open(path).convert("RGB")
        except Exception:
            img = PILImage.new("RGB", (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))
        if self.transform:
            img = self.transform(img)
        if self.has_labels:
            return img, int(row[self.label_col])
        return img

    @property
    def targets(self):
        if self.has_labels:
            return self.df[self.label_col].astype(int).tolist()
        return []

    @property
    def image_ids(self):
        return self.df[self.img_col].tolist()

# ── 1.4  Stratified 80/20 split from train.csv ────────────────────────────────
labels_all = train_df[LABEL_COL].values.astype(int)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(labels_all)), labels_all))

train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
val_split_df   = train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train split : {len(train_split_df)}  dist={Counter(train_split_df[LABEL_COL].tolist())}")
print(f"Val   split : {len(val_split_df)}  dist={Counter(val_split_df[LABEL_COL].tolist())}")
print(f"Test (challenge) : {len(test_df)} samples — labels unknown")

# ── 1.5  Build datasets ───────────────────────────────────────────────────────
train_dataset = ChallengeDataset(
    train_split_df, IMG_COL, LABEL_COL, train_transform, IMAGE_ROOT, mode="train")
val_dataset   = ChallengeDataset(
    val_split_df,   IMG_COL, LABEL_COL, val_transform,   IMAGE_ROOT, mode="val")
test_dataset  = ChallengeDataset(
    test_df, TEST_IMG_COL, None, val_transform, IMAGE_ROOT, mode="test")

# ── 1.6  DataLoaders ──────────────────────────────────────────────────────────
pin = torch.cuda.is_available()
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

# ── 1.7  Visualise one training batch ────────────────────────────────────────
imgs, lbls = next(iter(train_loader))
grid = torchvision.utils.make_grid(
    imgs[:16].cpu(), nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title(f"Sample training batch — labels: {lbls[:16].tolist()}")
plt.axis("off"); plt.tight_layout(); plt.show()

## Cell 2 — ConvNeXt V2 Tiny (First Backbone)

We replace EfficientNet-B0 with **ConvNeXt V2 Tiny** (`convnextv2_tiny.fcmae_ft_in22k_in1k`),
pre-trained on ImageNet-22k and fine-tuned on ImageNet-1k.  ConvNeXt V2 applies
**Fully Convolutional Masked Autoencoders (FCMAE)** self-supervised pre-training, which
forces the model to learn high-fidelity local texture representations — exactly the
signal needed to distinguish AI-generated from authentic images at >90% F1.

**Custom binary head (Cell 4-compatible):**
```
Linear(in_features → 256)  →  GELU  →  Dropout(0.3)  →  Linear(256 → 1)
```
> **Cell 4 embedding hook point:** `eff_model.classifier[1]` (GELU output, 256-d)  
> **Cell 8 Grad-CAM hook point:** `eff_model.features[-1][0]` (last stage, first block)

**Training schedule:**

| Phase | Unfrozen layers | Epochs | LR |
|-------|----------------|--------|-----|
| Warmup | Last 3 ConvNeXt stages + classifier | 5 | 1e-4 |
| Finetune | Same | 15 | 3e-5 |
| Auto full-backbone | Entire backbone (if val_F1 < 0.90) | 10 | 1e-5 |

Gradient clipping (max_norm=1.0) stabilises training with the stronger backbone.
AdamW + cosine annealing; best checkpoint saved by **validation F1**.

In [ ]:
# ── 2.1  Build ConvNeXt V2 Tiny with custom binary head ──────────────────────
class ConvNeXtBinaryModel(nn.Module):
    """
    ConvNeXt V2 Tiny backbone + custom 256-d binary head.

    Attribute layout designed for downstream compatibility:
      * self.classifier  – nn.Sequential used by Cell 4: eff_model.classifier[1]
      * self.features    – list[nn.Sequential] used by Cell 8: eff_model.features[-1][0]
    """
    def __init__(self):
        super().__init__()
        backbone = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in22k_in1k", pretrained=True, num_classes=0)
        in_features = backbone.num_features          # 768 for ConvNeXt V2 Tiny
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),  # [0] Linear
            nn.GELU(),                    # [1] ← Cell 4 embedding hook (256-d output)
            nn.Dropout(0.3),              # [2]
            nn.Linear(256, 1),            # [3]
        )
        # Cell 8 Grad-CAM: expose stage block sequences as a list.
        # backbone.stages is nn.Sequential of ConvNeXtStage; each stage has .blocks
        # (nn.Sequential of ConvNeXtBlock), which supports integer indexing [0].
        self.features = [s.blocks for s in backbone.stages]

    def forward(self, x):
        feats = self.backbone(x)       # (N, 768) pooled feature vector
        return self.classifier(feats)  # (N, 1) logit

# `eff_model` name is preserved intentionally: Cell 4 hooks eff_model.classifier[1]
# and Cell 8 uses eff_model.features[-1][0]. Renaming would break downstream cells.
eff_model = ConvNeXtBinaryModel().to(DEVICE)
total_p   = sum(p.numel() for p in eff_model.parameters())
print(f"ConvNeXt V2 Tiny  total params: {total_p:,}")

# ── 2.2  Training utilities (with gradient clipping + LR tracking) ────────────
def train_one_epoch(model, loader, optimizer, criterion, amp_scaler, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        amp_scaler.scale(loss).backward()
        if clip_grad > 0:
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    # Returns (avg_loss, f1, roc_auc, all_probs, all_labels)
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, all_probs, all_labels = 0.0, [], []
    for imgs, labels in loader:
        imgs     = imgs.to(DEVICE, non_blocking=True)
        labels_t = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels_t)
        total_loss += loss.item() * imgs.size(0)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    f1    = f1_score(all_labels, preds, zero_division=0)
    auc   = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
    return total_loss / len(loader.dataset), f1, auc, all_probs, all_labels


def run_training_phase(model, train_loader, val_loader, epochs, lr,
                       phase_name, ckpt_path, patience=5,
                       clip_grad=1.0, target_f1=0.0):
    """
    Generic training loop with:
      - AdamW optimiser + cosine annealing LR scheduler
      - Gradient clipping (clip_grad)
      - Early stopping: waits until best_f1 >= target_f1 OR patience*2 exceeded
      - LR history tracked in returned dict
    """
    criterion  = nn.BCEWithLogitsLoss()
    optimizer  = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    amp_scaler = GradScaler(enabled=USE_AMP)
    best_f1, no_improve = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_auc": [], "lr": []}

    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(current_lr)
        tr_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, amp_scaler, clip_grad)
        vl_loss, vl_f1, vl_auc, _, _ = evaluate(model, val_loader)
        scheduler.step()
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["val_f1"].append(vl_f1)
        history["val_auc"].append(vl_auc)
        print(f"[{phase_name}] Epoch {epoch:02d}/{epochs}  "
              f"lr={current_lr:.2e}  train_loss={tr_loss:.4f}  "
              f"val_loss={vl_loss:.4f}  val_F1={vl_f1:.4f}  val_AUC={vl_auc:.4f}")
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            # Stop only when patience exceeded AND target already met
            if no_improve >= patience and best_f1 >= target_f1:
                print(f"  Early stopping: best val_F1={best_f1:.4f} >= target={target_f1:.2f}")
                break
            # Hard stop to prevent infinite training
            if no_improve >= patience * 2:
                print(f"  Hard early stopping: best val_F1={best_f1:.4f}")
                break
    print(f"  Best val F1 after {phase_name}: {best_f1:.4f}")
    return history, best_f1

# ── 2.3  Warmup: unfreeze last 3 ConvNeXt stages + classifier ────────────────
for param in eff_model.backbone.parameters():
    param.requires_grad = False
for stage in eff_model.backbone.stages[-3:]:
    for param in stage.parameters():
        param.requires_grad = True
for param in eff_model.classifier.parameters():
    param.requires_grad = True

trainable_w = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
print(f"Warmup trainable: {trainable_w:,} / {total_p:,} ({100*trainable_w/total_p:.1f}%)")
print("=" * 60)
print("WARMUP — last 3 ConvNeXt stages + classifier (5 epochs, lr=1e-4)")
print("=" * 60)
hist_warmup, _ = run_training_phase(
    eff_model, train_loader, val_loader,
    epochs=5, lr=1e-4, phase_name="Warmup",
    ckpt_path="best_efficientnet.pth", patience=5, clip_grad=1.0)

# ── 2.4  Finetune: same unfrozen layers, lower LR ────────────────────────────
print("\n" + "=" * 60)
print("FINETUNE — last 3 stages + classifier (15 epochs, lr=3e-5)")
print("=" * 60)
hist_fine, _ = run_training_phase(
    eff_model, train_loader, val_loader,
    epochs=15, lr=3e-5, phase_name="Finetune",
    ckpt_path="best_efficientnet.pth", patience=5, clip_grad=1.0,
    target_f1=0.90)

# ── 2.5  Auto full-backbone unfreeze if val_F1 < 0.90 ─────────────────────────
eff_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
_, _cnx_f1_check, _, _, _ = evaluate(eff_model, val_loader)
print(f"\nBest val F1 after warmup+finetune: {_cnx_f1_check:.4f}")
hist_full = {"train_loss": [], "val_loss": [], "val_f1": [], "val_auc": [], "lr": []}

if _cnx_f1_check < 0.90:
    print("val_F1 < 0.90 — unfreezing FULL ConvNeXt backbone for additional fine-tuning...")
    for param in eff_model.backbone.parameters():
        param.requires_grad = True
    trainable_f = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
    print(f"Full backbone trainable: {trainable_f:,}")
    print("=" * 60)
    print("FULL-BACKBONE FINETUNE (10 epochs, lr=1e-5)")
    print("=" * 60)
    hist_full, _ = run_training_phase(
        eff_model, train_loader, val_loader,
        epochs=10, lr=1e-5, phase_name="FullFinetune",
        ckpt_path="best_efficientnet.pth", patience=5, clip_grad=0.5,
        target_f1=0.90)

# ── 2.6  Load best checkpoint + final evaluation ─────────────────────────────
eff_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
_, eff_val_f1, eff_val_auc, eff_val_probs, val_labels = evaluate(eff_model, val_loader)
print(f"\nConvNeXt V2 Tiny final  val_F1={eff_val_f1:.4f}  val_AUC={eff_val_auc:.4f}")
free_memory()

# ── 2.7  Training plots ───────────────────────────────────────────────────────
all_hist = {
    k: hist_warmup[k] + hist_fine[k] + hist_full.get(k, [])
    for k in hist_warmup
}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(all_hist["train_loss"], label="Train")
axes[0].plot(all_hist["val_loss"],   label="Val")
axes[0].set_title("ConvNeXt V2 — Loss"); axes[0].legend()
axes[1].plot(all_hist["val_f1"])
axes[1].axhline(0.90, color="red", linestyle="--", label="F1=0.90 target")
axes[1].set_title("ConvNeXt V2 — Val F1"); axes[1].legend()
axes[2].plot(all_hist["val_auc"]); axes[2].set_title("ConvNeXt V2 — Val AUC")
axes[3].plot(all_hist["lr"], color="purple")
axes[3].set_title("LR Schedule"); axes[3].set_yscale("log")
for ax in axes: ax.set_xlabel("Epoch")
plt.suptitle("ConvNeXt V2 Tiny Training", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

## Cell 3 — Swin Transformer Tiny (Second Backbone)

We replace ViT-B/16 with **Swin Transformer Tiny** (`swin_tiny_patch4_window7_224`),
pre-trained on ImageNet-1k.  Swin's hierarchical shifted-window self-attention
processes image patches at multiple resolutions — capturing both fine-grained
local texture artefacts and coarse semantic features that characterise AI-generated
images.  The hierarchical structure also reduces memory usage compared to full ViT,
making it well-suited for Colab T4 free tier.

**Custom binary head (Cell 4-compatible):**
```
Linear(768 → 256)  →  GELU  →  Dropout(0.3)  →  Linear(256 → 1)
```
> **Cell 4 embedding hook point:** `vit_model.head[1]` (GELU output, 256-d)  
> Checkpoint saved as `best_vit.pth` (downstream cells unchanged)

**Training schedule:**

| Phase | Unfrozen layers | Epochs | LR |
|-------|----------------|--------|-----|
| Finetune | Last 3 Swin stages + norm + head | 10 | 5e-5 |
| Extension (if val_F1 < 0.90) | Same | 10 | 1e-5 |

Swin's shifted-window attention improves detection of subtle patch-level synthetic
artefacts — over-smoothed regions, unnatural high-frequency textures, and inconsistent
lighting gradients — that require patch-level comparison to identify correctly.

In [ ]:
# ── 3.1  Build Swin Tiny with custom binary head ──────────────────────────────
# num_classes=0 → head becomes nn.Identity(); we replace it with our Sequential.
vit_model = timm.create_model(
    "swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)

in_features = vit_model.num_features   # 768 for Swin Tiny
vit_model.head = nn.Sequential(
    nn.Linear(in_features, 256),  # [0] Linear
    nn.GELU(),                    # [1] ← Cell 4 embedding hook (256-d output)
    nn.Dropout(0.3),              # [2]
    nn.Linear(256, 1),            # [3]
)
vit_model = vit_model.to(DEVICE)
total_sv  = sum(p.numel() for p in vit_model.parameters())
print(f"Swin Tiny  total params: {total_sv:,}")

# ── 3.2  Freeze all; unfreeze final 3 Swin stages + norm + head ──────────────
# Swin Tiny has 4 stages in vit_model.layers; unfreeze layers[-3:] = 1,2,3
for param in vit_model.parameters():
    param.requires_grad = False
for layer in vit_model.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True
for param in vit_model.norm.parameters():
    param.requires_grad = True
for param in vit_model.head.parameters():
    param.requires_grad = True

trainable_s = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f"Swin Tiny trainable: {trainable_s:,} / {total_sv:,} ({100*trainable_s/total_sv:.1f}%)")

# ── 3.3  Train Swin Tiny ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Swin Tiny fine-tuning (10 epochs, lr=5e-5)")
print("=" * 60)
swin_hist, swin_best_f1 = run_training_phase(
    vit_model, train_loader, val_loader,
    epochs=10, lr=5e-5, phase_name="Swin",
    ckpt_path="best_vit.pth", patience=5, clip_grad=1.0,
    target_f1=0.90)

# ── 3.4  Auto-extend if val_F1 < 0.90 ────────────────────────────────────────
vit_model.load_state_dict(torch.load("best_vit.pth", map_location=DEVICE))
_, _swin_f1_check, _, _, _ = evaluate(vit_model, val_loader)

if _swin_f1_check < 0.90:
    print(f"\nval_F1={_swin_f1_check:.4f} < 0.90 — extending training with lr=1e-5...")
    swin_hist_ext, _ = run_training_phase(
        vit_model, train_loader, val_loader,
        epochs=10, lr=1e-5, phase_name="Swin-Ext",
        ckpt_path="best_vit.pth", patience=5, clip_grad=1.0,
        target_f1=0.90)
    for k in swin_hist:
        swin_hist[k].extend(swin_hist_ext.get(k, []))

# ── 3.5  Load best Swin weights + final evaluation ───────────────────────────
vit_model.load_state_dict(torch.load("best_vit.pth", map_location=DEVICE))
_, vit_val_f1, vit_val_auc, vit_val_probs, _ = evaluate(vit_model, val_loader)
print(f"\nSwin Tiny final  val_F1={vit_val_f1:.4f}  val_AUC={vit_val_auc:.4f}")
free_memory()

# ── 3.6  Training plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(swin_hist["train_loss"], label="Train")
axes[0].plot(swin_hist["val_loss"],   label="Val")
axes[0].set_title("Swin Tiny — Loss"); axes[0].legend()
axes[1].plot(swin_hist["val_f1"])
axes[1].axhline(0.90, color="red", linestyle="--", label="F1=0.90 target")
axes[1].set_title("Swin Tiny — Val F1"); axes[1].legend()
axes[2].plot(swin_hist["val_auc"]); axes[2].set_title("Swin Tiny — Val AUC")
for ax in axes: ax.set_xlabel("Epoch")
plt.suptitle("Swin Transformer Tiny Training", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

## Cell 4 — Embedding Fusion + Meta-Learner Benchmark

Rather than stacking scalar probabilities, we extract **256-dimensional
penultimate-layer activations** from each backbone using forward hooks and
immediately transfer them to CPU NumPy arrays to minimise GPU memory pressure.

The two embedding vectors are concatenated into a **512-dimensional representation**:

```
EfficientNet  →  256-d embedding  ─┐
                                    ├── [512-d] → Meta-Learner → P(class=1)
ViT           →  256-d embedding  ─┘
```

We benchmark **three meta-learners**:

| # | Model | Key settings |
|---|-------|-------------|
| 1 | **XGBoost** (baseline) | max_depth=4, lr=0.03, n_estimators=300, subsample/colsample=0.9 |
| 2 | **CatBoost** | iterations=500, depth=6, lr=0.03, eval_metric=F1, auto_class_weights=Balanced |
| 3 | **TabPFN** | in-context learning transformer (try/except fallback) |

> **No leakage guarantee:** all meta-learners are fitted on **train embeddings only**.
> Validation embeddings are used solely for metric evaluation and threshold selection.
> The challenge test set is never seen during any fitting or tuning step.

In [ ]:
# ── 4.1  Install extra meta-learner packages ─────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=False)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=False)
    TABPFN_AVAILABLE = True
except Exception:
    TABPFN_AVAILABLE = False

# ── 4.2  Hook-based embedding extractor ──────────────────────────────────────
class EmbeddingExtractor:
    # Forward hook that captures penultimate-layer output on CPU immediately.
    def __init__(self, model, layer):
        self.embedding = None
        self._hook = layer.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        self.embedding = output.detach().cpu()

    def remove(self):
        self._hook.remove()

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.remove()


@torch.no_grad()
def extract_embeddings(model, loader, penultimate_layer):
    # Returns (embeddings[N,D], labels[N]|None, probs[N]) as CPU numpy arrays.
    model.eval()
    extractor = EmbeddingExtractor(model, penultimate_layer)
    all_embs, all_labels, all_probs = [], [], []
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            imgs, labels = batch
            all_labels.extend(labels.numpy().tolist())
        else:
            imgs = batch
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        all_embs.append(extractor.embedding.numpy())
        all_probs.extend(probs.tolist())
    extractor.remove()
    free_memory()
    embs       = np.vstack(all_embs)
    labels_out = np.array(all_labels) if all_labels else None
    return embs, labels_out, np.array(all_probs)


# ── 4.3  Identify penultimate layers ─────────────────────────────────────────
# EfficientNet classifier = [Linear(1280→256), ReLU, Dropout, Linear(256→1)]
#   hook on ReLU (index 1) captures 256-d output
eff_penultimate = eff_model.classifier[1]
# ViT head = [Linear(768→256), ReLU, Dropout, Linear(256→1)]
vit_penultimate = vit_model.head[1]

# ── 4.4  Extract embeddings (train / val / challenge-test) ────────────────────
print("Extracting EfficientNet embeddings...")
eff_train_emb, train_labels_emb, _ = extract_embeddings(eff_model, train_loader, eff_penultimate)
eff_val_emb,   val_labels_emb,   _ = extract_embeddings(eff_model, val_loader,   eff_penultimate)
eff_test_emb,  _,                _ = extract_embeddings(eff_model, test_loader,  eff_penultimate)

print("Extracting ViT embeddings...")
vit_train_emb, _, _ = extract_embeddings(vit_model, train_loader, vit_penultimate)
vit_val_emb,   _, _ = extract_embeddings(vit_model, val_loader,   vit_penultimate)
vit_test_emb,  _, _ = extract_embeddings(vit_model, test_loader,  vit_penultimate)

# ── 4.5  Concatenate → 512-d representation fusion ────────────────────────────
X_train = np.hstack([eff_train_emb, vit_train_emb])   # (N_train, 512)
X_val   = np.hstack([eff_val_emb,   vit_val_emb])     # (N_val,   512)
X_test  = np.hstack([eff_test_emb,  vit_test_emb])    # (N_test,  512)
print(f"Fused shapes  train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")

# StandardScaler fitted on train only — no leakage
feat_scaler = StandardScaler()
X_train = feat_scaler.fit_transform(X_train)
X_val   = feat_scaler.transform(X_val)
X_test  = feat_scaler.transform(X_test)
free_memory()

# ─────────────────────────────────────────────────────────────────────────────
# PART B — TRAIN THREE META-LEARNERS (all fitted on X_train / train_labels_emb)
# ─────────────────────────────────────────────────────────────────────────────

# ── 4.6  XGBoost baseline ────────────────────────────────────────────────────
print("\n" + "="*55 + "\nMETA-LEARNER 1 — XGBoost\n" + "="*55)
xgb_meta = xgb.XGBClassifier(
    max_depth=4, learning_rate=0.03, n_estimators=300,
    subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
    random_state=SEED, n_jobs=-1, tree_method="hist",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
xgb_meta.fit(X_train, train_labels_emb,
              eval_set=[(X_val, val_labels_emb)], verbose=50)
xgb_val_probs  = xgb_meta.predict_proba(X_val)[:, 1]
xgb_test_probs = xgb_meta.predict_proba(X_test)[:, 1]
print(f"XGBoost raw val AUC={roc_auc_score(val_labels_emb, xgb_val_probs):.4f}")

# ── 4.7  CatBoost ────────────────────────────────────────────────────────────
print("\n" + "="*55 + "\nMETA-LEARNER 2 — CatBoost\n" + "="*55)
from catboost import CatBoostClassifier
cat_meta = CatBoostClassifier(
    iterations=500, depth=6, learning_rate=0.03,
    eval_metric="F1", auto_class_weights="Balanced",
    early_stopping_rounds=50, random_seed=SEED, verbose=50,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)
cat_meta.fit(X_train, train_labels_emb, eval_set=(X_val, val_labels_emb))
cat_val_probs  = cat_meta.predict_proba(X_val)[:, 1]
cat_test_probs = cat_meta.predict_proba(X_test)[:, 1]
print(f"CatBoost raw val AUC={roc_auc_score(val_labels_emb, cat_val_probs):.4f}")

# ── 4.8  TabPFN ──────────────────────────────────────────────────────────────
print("\n" + "="*55 + "\nMETA-LEARNER 3 — TabPFN\n" + "="*55)
tabpfn_meta = tabpfn_val_probs = tabpfn_test_probs = None
tabpfn_best_thr = tabpfn_best_val_f1 = None
try:
    from tabpfn import TabPFNClassifier
    N_TABPFN = min(1000, len(X_train))
    rng = np.random.default_rng(SEED)
    idx_sub = rng.choice(len(X_train), N_TABPFN, replace=False)
    tabpfn_meta = TabPFNClassifier(device="cpu", N_ensemble_configurations=16)
    tabpfn_meta.fit(X_train[idx_sub], train_labels_emb[idx_sub])
    tabpfn_val_probs  = tabpfn_meta.predict_proba(X_val)[:, 1]
    tabpfn_test_probs = tabpfn_meta.predict_proba(X_test)[:, 1]
    print(f"TabPFN raw val AUC={roc_auc_score(val_labels_emb, tabpfn_val_probs):.4f}")
    TABPFN_AVAILABLE = True
except Exception as exc:
    print(f"TabPFN unavailable or failed ({exc}). Excluded from benchmark.")
    TABPFN_AVAILABLE = False

# ─────────────────────────────────────────────────────────────────────────────
# PART C — PER-LEARNER THRESHOLD OPTIMISATION (validation only)
# ─────────────────────────────────────────────────────────────────────────────
def optimise_threshold(val_probs, val_labels, low=0.20, high=0.80, steps=61):
    thresholds = np.linspace(low, high, steps)
    f1s = [f1_score(val_labels, (val_probs >= t).astype(int), zero_division=0)
           for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return float(thresholds[best_idx]), float(f1s[best_idx])

xgb_best_thr, xgb_best_val_f1 = optimise_threshold(xgb_val_probs, val_labels_emb)
cat_best_thr, cat_best_val_f1 = optimise_threshold(cat_val_probs, val_labels_emb)
print(f"XGBoost  best thr={xgb_best_thr:.2f}  val_F1={xgb_best_val_f1:.4f}")
print(f"CatBoost best thr={cat_best_thr:.2f}  val_F1={cat_best_val_f1:.4f}")
if TABPFN_AVAILABLE:
    tabpfn_best_thr, tabpfn_best_val_f1 = optimise_threshold(
        tabpfn_val_probs, val_labels_emb)
    print(f"TabPFN   best thr={tabpfn_best_thr:.2f}  val_F1={tabpfn_best_val_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# PART D — BENCHMARK OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
def learner_metrics(name, val_probs, val_labels, thr):
    preds = (val_probs >= thr).astype(int)
    tp = np.sum((preds == 1) & (val_labels == 1))
    fp = np.sum((preds == 1) & (val_labels == 0))
    fn = np.sum((preds == 0) & (val_labels == 1))
    return {
        "model":          name,
        "val_F1":         round(f1_score(val_labels, preds, zero_division=0), 4),
        "val_ROC_AUC":    round(roc_auc_score(val_labels, val_probs), 4),
        "val_PR_AUC":     round(average_precision_score(val_labels, val_probs), 4),
        "val_Precision":  round(tp / max(tp + fp, 1), 4),
        "val_Recall":     round(tp / max(tp + fn, 1), 4),
        "best_threshold": thr,
    }

rows = [
    learner_metrics("XGBoost",  xgb_val_probs, val_labels_emb, xgb_best_thr),
    learner_metrics("CatBoost", cat_val_probs, val_labels_emb, cat_best_thr),
]
if TABPFN_AVAILABLE:
    rows.append(learner_metrics("TabPFN", tabpfn_val_probs, val_labels_emb, tabpfn_best_thr))

benchmark_df = pd.DataFrame(rows).sort_values("val_F1", ascending=False).reset_index(drop=True)
print("\n── Benchmark Results ──")
print(benchmark_df.to_string(index=False))

# Bar plot
fig, ax = plt.subplots(figsize=(7, 4))
palette = ["#4C72B0", "#DD8452", "#55A868"]
ax.bar(benchmark_df["model"], benchmark_df["val_F1"],
       color=palette[:len(benchmark_df)], edgecolor="black")
ax.set_ylabel("Validation F1"); ax.set_title("Meta-Learner F1 Comparison")
ax.set_ylim(0, 1)
for i, v in enumerate(benchmark_df["val_F1"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

# ROC curves for all learners
all_val_probs = {"XGBoost": xgb_val_probs, "CatBoost": cat_val_probs}
if TABPFN_AVAILABLE:
    all_val_probs["TabPFN"] = tabpfn_val_probs
fig, ax = plt.subplots(figsize=(6, 5))
for (name, probs), col in zip(all_val_probs.items(), palette):
    fpr, tpr, _ = roc_curve(val_labels_emb, probs)
    ax.plot(fpr, tpr, color=col, lw=2,
            label=f"{name} (AUC={roc_auc_score(val_labels_emb, probs):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curves — Meta-Learner Comparison")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

# PR curves for all learners
fig, ax = plt.subplots(figsize=(6, 5))
for (name, probs), col in zip(all_val_probs.items(), palette):
    prec_c, rec_c, _ = precision_recall_curve(val_labels_emb, probs)
    ap = average_precision_score(val_labels_emb, probs)
    ax.plot(rec_c, prec_c, color=col, lw=2, label=f"{name} (AP={ap:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Meta-Learner Comparison")
ax.legend(loc="upper right"); plt.tight_layout(); plt.show()

# Identify best meta-learner
best_meta_name = benchmark_df.iloc[0]["model"]
best_meta_thr  = float(benchmark_df.iloc[0]["best_threshold"])
_meta_map = {
    "XGBoost":  (xgb_meta,  xgb_val_probs,    xgb_test_probs,    xgb_best_thr),
    "CatBoost": (cat_meta,  cat_val_probs,    cat_test_probs,    cat_best_thr),
}
if TABPFN_AVAILABLE:
    _meta_map["TabPFN"] = (tabpfn_meta, tabpfn_val_probs, tabpfn_test_probs, tabpfn_best_thr)

best_meta, best_meta_val_probs, best_meta_test_probs, best_meta_thr = _meta_map[best_meta_name]
print(f"\n★  Best: {best_meta_name}  "
      f"val_F1={benchmark_df.iloc[0]['val_F1']:.4f}  thr={best_meta_thr:.2f}")
free_memory()

## Cell 5 — Test-Time Augmentation (TTA)

We apply **5 deterministic views** of each image through the **best meta-learner**
selected in Cell 4 and average the probability estimates before thresholding.
TTA reduces prediction variance without any additional training.

| View | Transform |
|------|----------|
| 1 | Original (no augmentation) |
| 2 | Horizontal flip |
| 3 | Slight rotation ±10° |
| 4 | Colour jitter |
| 5 | Centre crop (90 %) → resize |

> TTA uses **pre-trained backbones + best meta-learner** at inference time only.

In [ ]:
# ── 5.1  TTA transforms ───────────────────────────────────────────────────────
tta_transforms = [
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
               T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomRotation(degrees=10),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)),
               T.CenterCrop(int(IMG_SIZE * 0.9)), T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
]


# ── 5.2  Single-transform embedding extraction ────────────────────────────────
@torch.no_grad()
def extract_emb_with_transform(model, df, img_col, label_col, transform,
                                penultimate_layer, img_root):
    # Build a temporary ChallengeDataset with the given transform and extract.
    tmp_ds = ChallengeDataset(df, img_col, label_col, transform, img_root)
    tmp_ldr = DataLoader(tmp_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS,
                         pin_memory=torch.cuda.is_available())
    return extract_embeddings(model, tmp_ldr, penultimate_layer)


# ── 5.3  Run 5-view TTA ───────────────────────────────────────────────────────
tta_val_probs_list  = []
tta_test_probs_list = []

for view_i, tfm in enumerate(tta_transforms, 1):
    print(f"TTA view {view_i}/5...")
    # Val
    e_v, _, _ = extract_emb_with_transform(
        eff_model, val_split_df, IMG_COL, LABEL_COL, tfm, eff_penultimate, IMAGE_ROOT)
    v_v, _, _ = extract_emb_with_transform(
        vit_model, val_split_df, IMG_COL, LABEL_COL, tfm, vit_penultimate, IMAGE_ROOT)
    X_v = feat_scaler.transform(np.hstack([e_v, v_v]))
    tta_val_probs_list.append(best_meta.predict_proba(X_v)[:, 1])

    # Test
    e_t, _, _ = extract_emb_with_transform(
        eff_model, test_df, TEST_IMG_COL, None, tfm, eff_penultimate, IMAGE_ROOT)
    v_t, _, _ = extract_emb_with_transform(
        vit_model, test_df, TEST_IMG_COL, None, tfm, vit_penultimate, IMAGE_ROOT)
    X_t = feat_scaler.transform(np.hstack([e_t, v_t]))
    tta_test_probs_list.append(best_meta.predict_proba(X_t)[:, 1])

tta_val_probs   = np.mean(tta_val_probs_list,  axis=0)
tta_test_probs  = np.mean(tta_test_probs_list, axis=0)
tta_val_labels  = np.array(val_split_df[LABEL_COL].tolist())

tta_val_f1 = f1_score(tta_val_labels, (tta_val_probs >= 0.5).astype(int), zero_division=0)
print(f"\nTTA val_F1 (thr=0.5): {tta_val_f1:.4f}")
free_memory()

## Cell 6 — Threshold Optimisation

The default decision threshold of 0.5 is rarely optimal for imbalanced binary
classification.  We sweep thresholds from **0.20 to 0.80** on the
**validation set only** and select the one maximising F1.

> **Leakage prevention:** the threshold is determined exclusively on the held-out
> validation split from `train.csv`.  The challenge `test.csv` is **never
> consulted** during this step — it is seen for the first time in Cell 9.

In [ ]:
# ── 6.1  Threshold sweep on TTA validation probabilities ─────────────────────
thresholds    = np.linspace(0.20, 0.80, 61)
val_f1_scores = np.array([
    f1_score(tta_val_labels, (tta_val_probs >= t).astype(int), zero_division=0)
    for t in thresholds
])
best_thr_idx      = int(np.argmax(val_f1_scores))
optimal_threshold = float(thresholds[best_thr_idx])
best_val_f1_opt   = float(val_f1_scores[best_thr_idx])

print(f"Optimal threshold : {optimal_threshold:.2f}")
print(f"Best val F1       : {best_val_f1_opt:.4f}")

# ── 6.2  Threshold vs F1 plot ─────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(thresholds, val_f1_scores, color="steelblue", linewidth=2)
plt.axvline(optimal_threshold, color="red", linestyle="--",
            label=f"Optimal={optimal_threshold:.2f}  F1={best_val_f1_opt:.3f}")
plt.xlabel("Decision threshold"); plt.ylabel("Validation F1")
plt.title(f"Threshold vs Validation F1  (TTA + {best_meta_name})")
plt.legend(); plt.tight_layout(); plt.show()

# ── 6.3  Apply optimal threshold ──────────────────────────────────────────────
final_val_preds  = (tta_val_probs  >= optimal_threshold).astype(int)
final_test_preds = (tta_test_probs >= optimal_threshold).astype(int)
print(f"Test class distribution: {Counter(final_test_preds.tolist())}")

## Cell 7 — Full Validation Evaluation

Comprehensive evaluation of the **best meta-learner + TTA + threshold-optimised pipeline**:

- Confusion matrix and classification report
- ROC curve (AUC)
- Precision-Recall curve (average precision)
- Feature importance from XGBoost / CatBoost (if applicable)
- False-positive image grid
- False-negative image grid
- McNemar's test vs EfficientNet standalone

**Why does embedding fusion improve results?**  
EfficientNet learns local texture and colour patterns through convolutional
receptive fields; ViT captures long-range global context via self-attention.
Combining both 256-d representations gives the meta-learner orthogonal information
that neither backbone alone provides, systematically reducing both FP and FN rates.

In [ ]:
# ── 7.1  Final validation metrics ────────────────────────────────────────────
final_val_f1  = f1_score(tta_val_labels, final_val_preds, zero_division=0)
final_val_auc = roc_auc_score(tta_val_labels, tta_val_probs)
print("=" * 50)
print(f"FINAL VAL  F1  : {final_val_f1:.4f}")
print(f"FINAL VAL  AUC : {final_val_auc:.4f}")
print("=" * 50)
print(classification_report(tta_val_labels, final_val_preds,
                             target_names=["Class 0", "Class 1"]))

# ── 7.2  Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(tta_val_labels, final_val_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title(f"Confusion Matrix — {best_meta_name} + TTA (val)")
plt.tight_layout(); plt.show()

# ── 7.3  ROC curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(tta_val_labels, tta_val_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC={final_val_auc:.3f})")
plt.plot([0, 1], [0, 1], "navy", linestyle="--")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"ROC Curve — {best_meta_name} + TTA")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

# ── 7.4  Precision-Recall curve ───────────────────────────────────────────────
prec_c, rec_c, _ = precision_recall_curve(tta_val_labels, tta_val_probs)
ap = average_precision_score(tta_val_labels, tta_val_probs)
plt.figure(figsize=(6, 5))
plt.plot(rec_c, prec_c, color="green", lw=2, label=f"AP={ap:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Precision-Recall — {best_meta_name} + TTA")
plt.legend(); plt.tight_layout(); plt.show()

# ── 7.5  Feature importance (XGBoost / CatBoost only) ────────────────────────
if best_meta_name in ("XGBoost", "CatBoost"):
    importances = (best_meta.feature_importances_
                   if best_meta_name == "XGBoost"
                   else best_meta.get_feature_importance())
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_vals  = importances[top20_idx]
    top20_names = [f"{'EffNet' if i < 256 else 'ViT'}-{i % 256}" for i in top20_idx]
    colors_fi   = ["steelblue" if i < 256 else "coral" for i in top20_idx]
    plt.figure(figsize=(10, 5))
    plt.bar(range(20), top20_vals, color=colors_fi)
    plt.xticks(range(20), top20_names, rotation=45, ha="right", fontsize=8)
    plt.ylabel("Feature importance")
    plt.title(f"Top-20 Feature Importances — {best_meta_name}")
    plt.legend(handles=[
        mpatches.Patch(color="steelblue", label="EfficientNet emb"),
        mpatches.Patch(color="coral",     label="ViT emb"),
    ]); plt.tight_layout(); plt.show()
else:
    print(f"Feature importance not plotted for {best_meta_name}.")

# ── 7.6  McNemar test: EfficientNet standalone vs full ensemble ───────────────
_, _, _, eff_val_probs_direct, eff_val_labels_direct = evaluate(eff_model, val_loader)
eff_val_preds_direct = (eff_val_probs_direct >= 0.5).astype(int)
both_c   = np.sum((eff_val_preds_direct == eff_val_labels_direct) & (final_val_preds == tta_val_labels))
eff_only = np.sum((eff_val_preds_direct == eff_val_labels_direct) & (final_val_preds != tta_val_labels))
ens_only = np.sum((eff_val_preds_direct != eff_val_labels_direct) & (final_val_preds == tta_val_labels))
both_w   = np.sum((eff_val_preds_direct != eff_val_labels_direct) & (final_val_preds != tta_val_labels))
cont     = np.array([[both_c, eff_only], [ens_only, both_w]])
mcn      = mcnemar(cont, exact=True)
print(f"\nMcNemar's Test (EfficientNet vs {best_meta_name}+TTA):")
print(f"  Contingency: {cont.tolist()}")
print(f"  p-value = {mcn.pvalue:.4f}  "
      f"({'significant' if mcn.pvalue < 0.05 else 'not significant'} at alpha=0.05)")

# ── 7.7  False-positive and false-negative image grids ────────────────────────
fp_idx = np.where((final_val_preds == 1) & (tta_val_labels == 0))[0]
fn_idx = np.where((final_val_preds == 0) & (tta_val_labels == 1))[0]
print(f"\nFalse positives: {len(fp_idx)}  |  False negatives: {len(fn_idx)}")

def show_error_grid(indices, title, n=8):
    if len(indices) == 0:
        print(f"No {title} samples."); return
    indices = indices[:n]
    fig, axes = plt.subplots(1, len(indices), figsize=(2*len(indices), 2.8))
    if len(indices) == 1: axes = [axes]
    for ax, idx in zip(axes, indices):
        img_id   = val_split_df.iloc[int(idx)][IMG_COL]
        img_path = resolve_image_path(str(img_id))
        try:
            img = PILImage.open(img_path).convert("RGB").resize((112, 112))
        except Exception:
            img = PILImage.new("RGB", (112, 112), color=(200, 200, 200))
        ax.imshow(img)
        ax.set_title(f"p={tta_val_probs[idx]:.2f}", fontsize=7)
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_error_grid(fp_idx, "False Positives (pred=1, true=0)")
show_error_grid(fn_idx, "False Negatives (pred=0, true=1)")
free_memory()

## Cell 8 — Grad-CAM + Error Analysis

**Gradient-weighted Class Activation Mapping (Grad-CAM)** highlights the image
regions most influential for each EfficientNet prediction.  We visualise TP, TN,
FP, and FN categories.

**Typical failure modes:**

| Category | Root cause |
|----------|-----------|
| False Positive | Misleading background texture resembling the positive class |
| False Positive | Handcrafted repetitive pattern confusion |
| False Negative | Severe lighting artefacts washing out discriminative features |
| False Negative | Multiple-object scenes: model attends to the wrong region |

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

# ── 8.1  Grad-CAM setup ───────────────────────────────────────────────────────
eff_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
eff_model.eval()

class _EfficientNetWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, x):
        return self.model(x).squeeze(1)

wrapped       = _EfficientNetWrapper(eff_model)
target_layers = [eff_model.features[-1][0]]
cam_engine    = GradCAM(model=wrapped, target_layers=target_layers)


def get_gradcam(img_id, label):
    path   = resolve_image_path(str(img_id))
    pil    = PILImage.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    rgb_np = np.array(pil, dtype=np.float32) / 255.0
    tensor = val_transform(pil).unsqueeze(0).to(DEVICE)
    cam    = cam_engine(input_tensor=tensor,
                        targets=[BinaryClassifierOutputTarget(label)])[0]
    return show_cam_on_image(rgb_np, cam, use_rgb=True), pil


def visualise_gradcam(indices, val_df, title, n=4):
    indices = [int(i) for i in indices[:n]]
    if not indices:
        print(f"No samples for: {title}"); return
    fig, axes = plt.subplots(2, len(indices), figsize=(3*len(indices), 6))
    if len(indices) == 1: axes = axes.reshape(2, 1)
    for col, idx in enumerate(indices):
        img_id = val_df.iloc[idx][IMG_COL]
        true_l = int(tta_val_labels[idx])
        pred_l = int(final_val_preds[idx])
        prob_v = float(tta_val_probs[idx])
        try:
            overlay, orig = get_gradcam(str(img_id), true_l)
        except Exception:
            overlay = orig = PILImage.new("RGB", (IMG_SIZE, IMG_SIZE), 200)
        axes[0, col].imshow(orig)
        axes[0, col].set_title(f"True:{true_l} Pred:{pred_l}\np={prob_v:.2f}",
                               fontsize=8)
        axes[0, col].axis("off")
        axes[1, col].imshow(overlay)
        axes[1, col].set_title("Grad-CAM", fontsize=8)
        axes[1, col].axis("off")
    fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.show()

# ── 8.2  Visualise all 4 prediction categories ────────────────────────────────
tp_idx = np.where((final_val_preds == 1) & (tta_val_labels == 1))[0]
tn_idx = np.where((final_val_preds == 0) & (tta_val_labels == 0))[0]

print("Grad-CAM — True Positives")
visualise_gradcam(tp_idx, val_split_df, "True Positives")
print("Grad-CAM — True Negatives")
visualise_gradcam(tn_idx, val_split_df, "True Negatives")
print("Grad-CAM — False Positives  (misleading texture / pattern confusion)")
visualise_gradcam(fp_idx, val_split_df,
    "False Positives — misleading background texture or handcrafted pattern confusion")
print("Grad-CAM — False Negatives  (lighting artefacts / multi-object ambiguity)")
visualise_gradcam(fn_idx, val_split_df,
    "False Negatives — lighting artefacts or multiple-object ambiguity")
free_memory()

## Cell 9 — Final Challenge Submission

We now apply the complete pipeline to the **official untouched `test.csv`**:

1. Best meta-learner (selected by validation F1 in Cell 4)
2. 5-view TTA (Cell 5)
3. Optimised decision threshold (Cell 6)
4. Generate `dcu_2026_final_submission.csv` preserving the **exact row order** of `test.csv`

> **This is the first and only time `test.csv` predictions are materialised into a file.**
> No test labels were observed at any prior step.
> The submission column names match those expected by `getting_started.ipynb`.

In [ ]:
# ── 9.1  Verify test set integrity ───────────────────────────────────────────
assert len(final_test_preds) == len(test_df), (
    f"Prediction count ({len(final_test_preds)}) != test.csv rows ({len(test_df)})"
)
print(f"Test predictions  : {len(final_test_preds)}")
print(f"Test image ids    : {len(test_df)}")
print(f"Missing ids       : {test_df[TEST_IMG_COL].isna().sum()}")
print(f"Predicted class dist: {Counter(final_test_preds.tolist())}")

# ── 9.2  Build submission DataFrame ──────────────────────────────────────────
# Column names preserved from getting_started.ipynb baseline convention
SUBMIT_ID_COL    = TEST_IMG_COL
SUBMIT_LABEL_COL = LABEL_COL

submission_df = pd.DataFrame({
    SUBMIT_ID_COL:    test_df[TEST_IMG_COL].values,
    SUBMIT_LABEL_COL: final_test_preds,
})

# ── 9.3  Save submission CSV ──────────────────────────────────────────────────
SUBMISSION_PATH = Path("/content/dcu_2026_final_submission.csv")
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved: {SUBMISSION_PATH}")

# ── 9.4  Verify and preview ───────────────────────────────────────────────────
check_df = pd.read_csv(SUBMISSION_PATH)
assert len(check_df) == len(test_df),    "Row count mismatch after saving!"
assert check_df[SUBMIT_ID_COL].isna().sum() == 0, "Missing image ids in submission!"
print(f"Row count OK : {len(check_df)}")
print(f"Columns      : {check_df.columns.tolist()}")
print(f"Label dist   : {check_df[SUBMIT_LABEL_COL].value_counts().to_dict()}")
print("\nFirst 10 rows:")
print(check_df.head(10).to_string(index=False))

# ── 9.5  Auto-download ────────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))
    print("\n Download triggered.")
except Exception:
    print(f"\nManually download from: {SUBMISSION_PATH}")

# ── 9.6  Submission summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUBMISSION SUMMARY")
print("=" * 60)
print(f"  Best meta-learner  : {best_meta_name}")
print(f"  Val F1 (TTA+thr)   : {best_val_f1_opt:.4f}")
print(f"  Decision threshold : {optimal_threshold:.2f}")
print(f"  TTA views          : 5")
print(f"  Submission rows    : {len(check_df)}")
print(f"  File               : {SUBMISSION_PATH}")
print("=" * 60)
print("\n✓ Notebook complete — ready for competition submission.")